# 1D CNN-A — Cluster Quality: How Many Clusters?

Evaluates K-Means cluster quality across K = 2 … 16 using three complementary
metrics: the elbow method (inertia), silhouette score, and Davies-Bouldin index.
Use this to validate — or update — the `N_CLUSTERS` value in `config.py`.

> **Prerequisite:** run `1dcnn_train.ipynb` first — it saves `model.pt` to
> `DATA_DIR / SYMBOL /`.

## 1. Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from config import Config
from data import (
    load_bars, clean_data, add_features, drop_feature_nans,
    scale_features, make_windows, filter_gap_windows,
)
from model import ConvAutoencoder, WindowDataset, load_model

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

In [ ]:
from config import Config

cfg = Config()

# ── Override defaults here before running the rest of the notebook ────────────
# cfg.MAX_BARS       = None   # load all bars (~552k)
# cfg.EPOCHS         = 30     # full training run
# cfg.N_CLUSTERS     = 12     # try more/fewer clusters
# cfg.LATENT_DIM     = 64     # larger latent space
# cfg.N_SAMPLE       = 5_000  # render more windows in Section 9

# Expose all config fields as module-level names so every downstream cell
# can use SYMBOL, WINDOW_SIZE, LR, feature_cols, DEVICE, etc. unchanged.
globals().update(vars(cfg))

print(f"Symbol={SYMBOL}  Timeframe={TIMEFRAME}  {START_DATE} → {END_DATE}")
print("Using device:", DEVICE)

## 3. Fetch Data from Alpaca API
Calls the local `alpaca_api` FastAPI service (must be running: `uv run main.py`).
Fetches TSLA 1-minute bars and saves to both DB and CSV.

In [ ]:
if FETCH_DATA:
    import httpx  # only needed when FETCH_DATA = True
    params = {
        "symbols": SYMBOL,
        "timeframe": TIMEFRAME,
        "start": START_DATE,
        "end": END_DATE,
        "save_to": "db,csv",
    }
    with httpx.Client(timeout=None) as client:
        r = client.get(f"{API_BASE}/bars", params=params)
        r.raise_for_status()
        result = r.json()
    bars = result.get("data", {}).get("bars", {}).get(SYMBOL, [])
    print(f"Fetched {len(bars)} bars for {SYMBOL}")
    print("Saved:", result.get("saved"))
else:
    print("FETCH_DATA=False — skipping. Set True in Config to re-pull.")

## 4. Load Data

In [ ]:
# Load raw OHLCV bars from the CSV file.
df = load_bars(DATA_DIR, SYMBOL, TIMEFRAME, MAX_BARS)

Check for:

Duplicate timestamps
Missing timestamps (gaps)
NaNs
Infinite values
Bad OHLC relationships (high < low, etc.)

Typical checks:

In [ ]:
# Remove duplicate timestamps and rows with missing values.
df = clean_data(df)

## 5. Verify Time Continuity
A CNN assumes a consistent sequence.

Look for:

missing bars
duplicate bars
irregular spacing

If you're using 1-minute candles, every row should be exactly 1 minutes apart.

In [ ]:
delta = df["timestamp"].diff()
dt = pd.to_timedelta(delta).dt.total_seconds()
print("Average time delta (seconds):", dt.mean())
print("Time delta distribution (seconds):")
print(dt.describe())

## 6. Add Features

In [ ]:
# Calculate 14 technical indicator columns (EMAs, MACD, candle shape, returns, volume ratio).
df = add_features(df)

## 6. Remove Initial NaNs

Feature engineering creates NaNs.

In [ ]:
# Drop the warm-up rows where EMAs and rolling means don't have enough history yet.
df = drop_feature_nans(df)

## 7. Scale Features

This is critical.

CNNs train poorly on:

close = 45000
volume = 10000000
return = 0.001

all mixed together.

StandardScaler

Most common:

In [ ]:
# Normalise every feature column so they all sit in a similar numeric range.
# RobustScaler uses the median and IQR — better than mean/std for financial data with outliers.
df, scaler = scale_features(df, feature_cols)

## 8. Create Fixed-Length Windows

A CNN does not ingest an entire dataframe.

It ingests samples.

In [ ]:
# Slice the time series into overlapping WINDOW_SIZE-bar windows.
# Each window is one training sample for the CNN.
X_raw = make_windows(df, feature_cols, WINDOW_SIZE)
n_features = len(feature_cols)

## 11. Filter Gap Windows
A window that spans an overnight or weekend gap mixes pre-gap and post-gap bars — the CNN would learn noise, not patterns. Any window whose 64-bar span crosses a gap > 5 minutes is dropped.

In [ ]:
# Remove windows that span overnight or weekend gaps.
# Such windows would teach the model noise rather than real patterns.
X_clean, valid_mask = filter_gap_windows(X_raw, df, WINDOW_SIZE)

## 13. Autoencoder Model
Encoder compresses `(batch, 14, 64)` → latent vector `(batch, LATENT_DIM)`.
Decoder reconstructs `(batch, 14, 64)` from the latent vector.
Training loss is reconstruction MSE — no labels needed.

In [ ]:
# ConvAutoencoder is defined in model.py — import it at the top.
# Here we just create an instance and move it to the device (CPU or GPU).
model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

In [ ]:
# WindowDataset is defined in model.py and imported at the top.
# It wraps a tensor so PyTorch's DataLoader can iterate it in batches.
# (No code needed here — the import at the top handles it.)

In [ ]:
# Load the weights saved by 1dcnn_train.ipynb.
# load_model() builds an empty ConvAutoencoder and fills it with the saved weights.
model = load_model(DATA_DIR, SYMBOL, n_features, LATENT_DIM, DEVICE)

## 15. Extract Latent Vectors

In [ ]:
all_loader = DataLoader(WindowDataset(
    torch.tensor(X_clean).permute(0, 2, 1)
), batch_size=BATCH_SIZE, shuffle=False)

model.eval()
Z_list = []
with torch.no_grad():
    for batch in all_loader:
        Z_list.append(model.encoder(batch.to(DEVICE)).cpu().numpy())

Z = np.concatenate(Z_list)   # (N_clean, LATENT_DIM)
print(f'Latent matrix Z: {Z.shape}')

## 18. Cluster Quality Metrics

Three metrics, each measuring a different aspect of cluster quality:

| Metric | What it measures | Best value |
|--------|-----------------|------------|
| **Inertia (elbow)** | Total within-cluster variance — lower = tighter clusters | Look for the *bend* (elbow) |
| **Silhouette score** | How similar each point is to its own cluster vs. neighbours | Higher = better (max 1.0) |
| **Davies-Bouldin index** | Average ratio of within-cluster scatter to between-cluster distance | Lower = better (min 0.0) |

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

K_RANGE = range(2, 17)
inertias, silhouettes, db_scores = [], [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(Z)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(Z, labels, sample_size=5000, random_state=42))
    db_scores.append(davies_bouldin_score(Z, labels))
    print(f'  K={k:2d}  inertia={km.inertia_:,.0f}  silhouette={silhouettes[-1]:.4f}  DB={db_scores[-1]:.4f}')

k_vals = list(K_RANGE)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Elbow (inertia)
ax = axes[0]
ax.plot(k_vals, inertias, 'o-', color='royalblue', lw=2)
ax.set_title('Elbow Method — Inertia', fontsize=12)
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('Inertia (within-cluster sum of squares)')
ax.grid(alpha=0.3)

# 2. Silhouette
best_sil_k = k_vals[int(np.argmax(silhouettes))]
ax = axes[1]
ax.plot(k_vals, silhouettes, 'o-', color='seagreen', lw=2)
ax.axvline(best_sil_k, color='tomato', ls='--', lw=1.5, label=f'peak K={best_sil_k}')
ax.set_title('Silhouette Score (higher = better)', fontsize=12)
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('Silhouette score')
ax.legend(); ax.grid(alpha=0.3)

# 3. Davies-Bouldin
best_db_k = k_vals[int(np.argmin(db_scores))]
ax = axes[2]
ax.plot(k_vals, db_scores, 'o-', color='darkorange', lw=2)
ax.axvline(best_db_k, color='tomato', ls='--', lw=1.5, label=f'best K={best_db_k}')
ax.set_title('Davies-Bouldin Index (lower = better)', fontsize=12)
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('Davies-Bouldin index')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(f'Cluster Quality Metrics — TSLA {TIMEFRAME} Latent Space', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Silhouette peak at K={best_sil_k}')
print(f'Davies-Bouldin best at K={best_db_k}')
print(f'Current config N_CLUSTERS={N_CLUSTERS}')

## 19. Recommendation

### How to pick K
The three metrics rarely all agree. Use them together:

1. **Elbow** — find where the inertia curve bends and stops dropping steeply.
   The K at the bend is where you get most of the benefit of clustering.

2. **Silhouette** — the peak K means windows are most clearly separated from
   their neighbours. Prefer this if you want well-defined, distinct clusters.

3. **Davies-Bouldin** — the minimum K means clusters are tight relative to
   their distance from each other. Agrees with silhouette when K is a clear choice.

**If all three point to the same K:** that's your answer — update `N_CLUSTERS` in `config.py`.

**If they disagree:** the data doesn't have a single obvious cluster count. Choose
the K that makes the most interpretable `latent_cluster.ipynb` centroid plots.